### Importing libraries

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.io import savemat
from scipy.signal import welch
import mne

output_dir = Path("../data/processed")
THETA_BAND = (4, 8)



### Making a function to extract the theta power

In [ ]:
def extract_theta_power(epoch_data, band=THETA_BAND):
     """Extracts the theta power from the given epoch data using Welch's method.
     
     Parameters:
     epoch_data (epo.fif): The epoch data being extracted
     band (tuple): The frequency band to be extracted, default is theta band of (4, 8)
     
     Returns: 
          average (numpy.ndarray): The average theta power across the epoch
     """
     
     # Loading the epochs
     epochs = mne.read_epochs(epoch_data, preload=True, verbose=False)
     spectrum = epochs.compute_psd(method='welch', fmin=1, fmax=40)
     psds, freqs = spectrum.get_data(return_freqs=True) # psds shape is (n_epochs, n_channels, n_freqs)
     band_mask = (freqs >= band[0]) & (freqs <= band[1])
     # average the psd across the theta band 
     average = psds[:, :, band_mask].mean(axis=(1,2)) # shape is (n_epochs, n_channels)
     return average


### Example Theta-Power Table

In [ ]:
rows = []
epoch_files = sorted(output_dir.glob("*.fif"))
print(f"Processed segment files #{len(epoch_files)}")

for file in epoch_files:
     # format reminder: subject_##_test#_phase#-epo.fif
     stem = file.stem.replace("-epo", "")
     parts = stem.split("_")
     subject = f"{parts[0]}_{parts[1]}"
     test = int(parts[2].replace("test", ""))
     phase = int(parts[3].replace("phase", ""))
     
     theta_power = extract_theta_power(file)
     for p in theta_power:
          rows.append([subject, test, phase, p])

features_df = pd.DataFrame(rows, columns=["subject", "test", "phase", "theta_power"])
features_df.to_csv(output_dir / "theta_power_features.csv", index=True)
print(f"Saved theta power features to theta_power_features.csv with shape {features_df.shape}")
features_df.groupby("test")["theta_power"].describe()


### Export one epoch to check against MATLAB

In [ ]:
example_path = output_dir / "subject_07_test1_phase2-epo.fif"
example_epochs = mne.read_epochs(example_path, preload=True)
sfreq = example_epochs.info['sfreq']

example_signal = example_epochs.get_data()[0].mean(axis=0) # takes just the first epoch
f, pxx = welch(example_signal, fs=sfreq, nperseg=128)
# 256 to maximum of the signal's length in order to prevent errors

python_theta_power = np.trapezoid(pxx[(f >= THETA_BAND[0]) & (f <= THETA_BAND[1])], f[(f >= THETA_BAND[0]) & (f <= THETA_BAND[1])])

print(f"Example signal length: {len(example_signal)} is sampled at {sfreq} Hz")
print(f"Python calculated theta power (welch): {python_theta_power:.4e}")

savemat(output_dir / "matlab_check.mat", {
     "example_signal": example_signal,
     "sfreq": sfreq,
     "python_theta_power": python_theta_power
})  

print("Finished making and saving matlab_check.mat")

### Load MATLAB results into Python

In [ ]:
anova_results = pd.read_csv(output_dir / "matlab_anova_results.csv")
anova_results

### Marking Results

Python theta power: 1.9655e-12
MATLAB: 2.750e-12
Rel Diff: 39.92%
p-value: 4.756163e-75

Yes it did because the ANOVA p-value was less than 0.05
which means that the difference in theta power across workload levels was statistically significant and did not occur due to chance.